# CS781 Phase 4 · analysis only (no training)

In [ ]:
import os, sys, glob, json
from pathlib import Path
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
os.makedirs(f"{WORK}/p4cjpe", exist_ok=True); sys.path.insert(0, WORK)
cands = [os.environ.get("P4_RUNS", "")] + [str(p.parent.parent) for p in Path("/kaggle/input").rglob("summary.json")
                                          if p.parent.parent.name == "runs"] if os.path.exists("/kaggle/input") else [os.environ.get("P4_RUNS", "")]
cands = [c for c in dict.fromkeys(cands) if c and os.path.isdir(c)]
assert cands, "no runs/ folder found: attach the finished Phase-4 notebook's output (Add Input -> Notebook output)"
OUT_DIR = cands[0]
ROOT = os.path.dirname(OUT_DIR)
print("runs:", OUT_DIR); print(sorted(os.listdir(OUT_DIR)))

In [ ]:
%%writefile {WORK}/p4cjpe/__init__.py
"""CS781 Phase 4: Court Judgment Prediction (CJPE) on the Phase-1 legal language models."""

In [ ]:
%%writefile {WORK}/p4cjpe/metrics.py
"""Binary judgment-prediction metrics (Codabench: macro-F1 primary; precision and recall secondary) and the
dev-set decision-threshold search used for calibration."""
import numpy as np

def binary_report(gold, pred):
    g, p = np.asarray(gold, dtype=np.int64), np.asarray(pred, dtype=np.int64)
    out = {"per_class": {}}
    for k in (0, 1):
        tp = int(((p == k) & (g == k)).sum())
        fp = int(((p == k) & (g != k)).sum())
        fn = int(((p != k) & (g == k)).sum())
        P = tp / (tp + fp) if tp + fp else 0.0
        R = tp / (tp + fn) if tp + fn else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        out["per_class"][k] = {"p": P, "r": R, "f1": F, "gold": int((g == k).sum()), "pred": int((p == k).sum())}
    pc = out["per_class"]
    out.update(macro_f1=(pc[0]["f1"] + pc[1]["f1"]) / 2, macro_p=(pc[0]["p"] + pc[1]["p"]) / 2,
               macro_r=(pc[0]["r"] + pc[1]["r"]) / 2, accuracy=float((g == p).mean()) if len(g) else 0.0,
               n=int(len(g)))
    out["confusion"] = [[int(((g == a) & (p == b)).sum()) for b in (0, 1)] for a in (0, 1)]
    try:
        from sklearn.metrics import f1_score
        out["sklearn_macro_f1"] = float(f1_score(g, p, average="macro", labels=[0, 1], zero_division=0))
    except ImportError:
        pass
    return out

def best_threshold(gold, prob1, lo=0.05, hi=0.95, step=0.01):
    g = np.asarray(gold)
    best = (-1.0, 0.5)
    for t in np.arange(lo, hi + 1e-9, step):
        f = binary_report(g, (np.asarray(prob1) >= t).astype(int))["macro_f1"]
        if f > best[0] + 1e-12 or (abs(f - best[0]) <= 1e-12 and abs(t - 0.5) < abs(best[1] - 0.5)):
            best = (f, float(round(t, 4)))
    return best[1], best[0]

def format_report(rep, title=""):
    pc = {int(k): v for k, v in rep["per_class"].items()}
    lines = [title] if title else []
    lines.append(f"{'class':8s} {'P':>6s} {'R':>6s} {'F1':>6s} {'gold':>6s} {'pred':>6s}")
    for k in (0, 1):
        lines.append(f"{k:<8d} {pc[k]['p']:6.3f} {pc[k]['r']:6.3f} {pc[k]['f1']:6.3f} {pc[k]['gold']:6d} {pc[k]['pred']:6d}")
    lines.append(f"{'MACRO':8s} {rep['macro_p']:6.3f} {rep['macro_r']:6.3f} {rep['macro_f1']:6.3f}   (accuracy {rep['accuracy']:.3f}, "
                 f"{rep['n']} cases, confusion [[TN FP] [FN TP]] = {rep['confusion']})")
    return "\n".join(lines)

In [ ]:
%%writefile {WORK}/p4cjpe/data.py
"""Phase-4 data: download, load, validate, tokenise once (cached), statistics.

Layout (HF dataset Exploration-Lab/CS-781-Capstone, folder Phase_4_CJPE):
    Train/train.json, Train/dev.json, Test/test.json
A case is {"id": "case_001", "text": "...", "label": 0 | 1} (no label in test.json).
The loader also accepts a dict {id: {...}}, JSON lines, a {"data": [...]} wrapper, "text" given as a list of
paragraphs, and the label under "label"/"labels"/"y"/"outcome"/... so a small format difference does not break it.

Tokenising ~0.8 GB of judgments with the pure-Python Phase-1 BPE takes a few minutes, so it is done ONCE
(multiprocessing) and stored as uint16 token arrays in cache_dir; every run (and both parallel GPU jobs) reuses it.
Cached form per split: {"ids": [str], "raw_ids": [original id], "labels": np.int64 array | None,
                        "tokens": [np.uint16 array], "n_chars": np.int64 array}
"""
import hashlib
import json
import os
import pickle
from collections import Counter
from pathlib import Path

import numpy as np

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_SUBDIR = "Phase_4_CJPE"
SPLITS = ("train", "dev", "test")
ID_KEYS = ("id", "case_id", "doc_id", "uid", "name", "case")
TEXT_KEYS = ("text", "document", "doc", "case_text", "judgment", "content", "body")
LABEL_KEYS = ("label", "labels", "y", "outcome", "judgment_label", "decision", "target", "pred")

def download(local_dir, token=None):
    from huggingface_hub import snapshot_download
    token = token or os.environ.get("HF_TOKEN")
    snapshot_download(repo_id=HF_REPO, repo_type="dataset", local_dir=local_dir, token=token,
                      allow_patterns=[f"{HF_SUBDIR}/*"])
    return Path(local_dir) / HF_SUBDIR

def find_data_dir(candidates=()):
    roots = [Path(c) for c in candidates if c] + [Path("/kaggle/input")]
    for r in roots:
        if not r.exists():
            continue
        if (r / "Train" / "train.json").exists() and (r / "Test" / "test.json").exists():
            return r
        for p in r.rglob("train.json"):
            if p.parent.name == "Train" and (p.parent.parent / "Test" / "test.json").exists():
                return p.parent.parent
    return None

def split_path(data_dir, split):
    return Path(data_dir) / ("Test" if split == "test" else "Train") / f"{split}.json"

def _read_json(path):
    txt = Path(path).read_text(encoding="utf-8")
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        return [json.loads(line) for line in txt.splitlines() if line.strip()]

def _records(raw):
    if isinstance(raw, dict):
        for k in ("data", "documents", "docs", "cases", "examples"):
            if isinstance(raw.get(k), list):
                return raw[k]
        recs = []
        for k, v in raw.items():
            if isinstance(v, dict):
                recs.append({"id": k, **v} if not any(x in v for x in ID_KEYS) else v)
            elif isinstance(v, str):
                recs.append({"id": k, "text": v})
            else:
                raise ValueError(f"unexpected value type for case {k}: {type(v)}")
        return recs
    if isinstance(raw, list):
        return raw
    raise ValueError(f"unexpected top-level JSON type {type(raw)}")

def _first(r, keys):
    for k in keys:
        if k in r and r[k] is not None:
            return k, r[k]
    return None, None

def _label(v, where):
    if isinstance(v, list) and len(v) == 1:
        v = v[0]
    if isinstance(v, bool):
        return int(v)
    if isinstance(v, (int, np.integer)) and int(v) in (0, 1):
        return int(v)
    if isinstance(v, float) and v in (0.0, 1.0):
        return int(v)
    if isinstance(v, str) and v.strip() in ("0", "1"):
        return int(v.strip())
    raise ValueError(f"{where}: label {v!r} is not 0/1")

def normalise_text(t):
    if isinstance(t, list):
        t = " ".join("" if x is None else str(x) for x in t)
    return " ".join(str(t).split())

def norm_case(r, where, need_label):
    if not isinstance(r, dict):
        raise ValueError(f"{where}: record is {type(r)}, expected an object")
    _, rid = _first(r, ID_KEYS)
    _, text = _first(r, TEXT_KEYS)
    lk, lab = _first(r, LABEL_KEYS)
    if rid is None:
        raise ValueError(f"{where}: case without an id field: keys {sorted(r)}")
    if text is None:
        raise ValueError(f"{where}: case {rid} has no text field: keys {sorted(r)}")
    if need_label and lab is None:
        raise ValueError(f"{where}: case {rid} has no label field: keys {sorted(r)}")
    return {"id": str(rid), "raw_id": rid, "text": normalise_text(text),
            "label": None if (lab is None or not need_label) else _label(lab, f"{where}/{rid}")}

def load_split(data_dir, split, max_docs=None):
    p = split_path(data_dir, split)
    if not p.exists():
        raise FileNotFoundError(p)
    recs = _records(_read_json(p))
    if max_docs:
        recs = recs[:max_docs]
    cases = [norm_case(r, f"{p.name}[{i}]", split != "test") for i, r in enumerate(recs)]
    seen, out, dup, conflict = {}, [], 0, 0
    for c in cases:
        if c["id"] in seen:
            dup += 1
            if split == "test":
                raise ValueError(f"test.json: id {c['id']} appears twice; one prediction per id is required")
            conflict += int(seen[c["id"]]["label"] != c["label"])
            continue
        seen[c["id"]] = c
        out.append(c)
    if dup:
        print(f"[data] {split}: dropped {dup} repeated ids ({conflict} with a different label; first one kept)")
    return out

def describe_raw(data_dir, n_chars=300):
    for split in SPLITS:
        p = split_path(data_dir, split)
        raw = _read_json(p)
        recs = _records(raw)
        first = recs[0] if recs else {}
        short = {k: (v[:n_chars] + "...") if isinstance(v, str) and len(v) > n_chars else v for k, v in first.items()}
        print(f"{p.name}: {p.stat().st_size / 2**20:.1f} MB | top-level {type(raw).__name__} | {len(recs)} records | "
              f"keys {sorted(first)} | {json.dumps(short, default=str)[:n_chars + 200]}")

_TOK = None

def _init_worker(tok_path):
    global _TOK
    from .tokenizer import BPETokenizer
    _TOK = BPETokenizer.load(tok_path)

def _encode(text):
    ids = _TOK.encode(" " + text) if text else []
    if not ids:
        ids = _TOK.encode(" .")
    return np.asarray(ids, dtype=np.uint16)

def tokenizer_fingerprint(tok_path):
    from .tokenizer import BPETokenizer
    tok = BPETokenizer.load(tok_path)
    assert tok.vocab_size <= 65535, "uint16 token cache needs vocab <= 65535"
    return tok.fingerprint()

def cache_path(cache_dir, split, tok_path, max_docs=None):
    tag = tokenizer_fingerprint(tok_path) + (f"_n{max_docs}" if max_docs else "")
    return Path(cache_dir) / f"cjpe_{split}_{tag}.pkl"

def tokenise_split(cases, tok_path, n_proc=4, log=print):
    texts = [c["text"] for c in cases]
    if n_proc and n_proc > 1 and len(texts) > 64:
        import gc
        import multiprocessing as mp
        gc.collect(); gc.freeze()
        try:
            with mp.get_context("fork").Pool(n_proc, initializer=_init_worker, initargs=(tok_path,)) as pool:
                toks = list(pool.imap(_encode, texts, chunksize=32))
        finally:
            gc.unfreeze()
    else:
        _init_worker(tok_path)
        toks = [_encode(t) for t in texts]
    return toks

def build_cache(data_dir, tok_path, cache_dir, n_proc=4, max_docs=None, log=print, splits=SPLITS):
    import time
    Path(cache_dir).mkdir(parents=True, exist_ok=True)
    for split in splits:
        cp = cache_path(cache_dir, split, tok_path, max_docs)
        if cp.exists():
            continue
        t0 = time.time()
        cases = load_split(data_dir, split, max_docs)
        toks = tokenise_split(cases, tok_path, n_proc, log)
        obj = {"ids": [c["id"] for c in cases], "raw_ids": [c["raw_id"] for c in cases],
               "labels": None if split == "test" else np.array([c["label"] for c in cases], dtype=np.int64),
               "tokens": toks, "n_chars": np.array([len(c["text"]) for c in cases], dtype=np.int64),
               "source": str(split_path(data_dir, split))}
        tmp = cp.with_suffix(f".{os.getpid()}.tmp")
        with open(tmp, "wb") as f:
            pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, cp)
        n_tok = sum(len(t) for t in toks)
        log(f"[data] {split}: {len(cases)} cases, {n_tok / 1e6:.1f}M tokens, tokenised in {time.time() - t0:.0f}s -> {cp.name}")
        del cases, toks, obj

def load_cached(data_dir, tok_path, cache_dir, max_docs=None, n_proc=4, log=print):
    build_cache(data_dir, tok_path, cache_dir, n_proc, max_docs, log)
    out = {}
    for split in SPLITS:
        with open(cache_path(cache_dir, split, tok_path, max_docs), "rb") as f:
            out[split] = pickle.load(f)
    return out

def text_fingerprints(cases):
    return {hashlib.sha1(c["text"].encode("utf-8")).hexdigest(): c["id"] for c in cases}

def stats(data):
    out = {}
    for split in SPLITS:
        d = data[split]
        lens = np.array([len(t) for t in d["tokens"]])
        rec = {"cases": len(lens), "tokens": {q: int(np.percentile(lens, p)) for q, p in
                                              (("min", 0), ("p10", 10), ("p50", 50), ("mean", None), ("p90", 90), ("max", 100))
                                              if p is not None} if len(lens) else {}}
        if len(lens):
            rec["tokens"]["mean"] = int(lens.mean())
            rec["chars_per_token"] = round(float(d["n_chars"].sum() / max(1, lens.sum())), 2)
            for L in (512, 2048, 4096):
                rec[f"fits_in_{L}"] = round(float((lens <= L).mean()), 3)
        if d["labels"] is not None:
            c = Counter(d["labels"].tolist())
            rec["labels"] = {0: c.get(0, 0), 1: c.get(1, 0)}
            rec["share_label_1"] = round(c.get(1, 0) / max(1, len(lens)), 3)
            if len(lens):
                rec["median_tokens_by_label"] = {k: int(np.median(lens[d["labels"] == k])) if (d["labels"] == k).any() else 0
                                                 for k in (0, 1)}
        out[split] = rec
    tr = set(data["train"]["ids"])
    out["id_overlap"] = {"train&dev": len(tr & set(data["dev"]["ids"])), "train&test": len(tr & set(data["test"]["ids"]))}
    return out

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
for m in [m for m in list(sys.modules) if m.startswith("p4cjpe")]:
    del sys.modules[m]
from p4cjpe import data as D
from p4cjpe.metrics import format_report
SUMS = {json.load(open(p))["run"]: json.load(open(p)) for p in sorted(glob.glob(f"{OUT_DIR}/*/summary.json"))}
rows = []
for r, s in SUMS.items():
    m, ch = s["method"], s["chunks"]
    rows.append({"run": r, "arch": s["arch"], "backbone": s["backbone"], "layers": s["layers"],
                 "method": m["name"] + ("+KD" if s.get("kd_teacher") else ""), "doc_repr": f"{ch['k']}x{ch['len']} {ch['strategy']}",
                 "params_M": round(s["params_total"] / 1e6, 1), "trainable_M": round(s["params_trainable"] / 1e6, 3),
                 "trainable_%": s["trainable_pct"], "train_min": s["train_minutes"], "epochs": s["epochs_run"], "best_epoch": s["best_epoch"],
                 "peak_mem_GB": round(s["peak_train_mem_gb"] or 0, 2), "docs_per_s": s["infer_docs_per_sec"],
                 "dev_F1@0.5": round(s["dev"]["macro_f1"], 4), "thr": s["threshold"],
                 "dev_macroF1": round(s["dev_calibrated"]["macro_f1"], 4), "dev_macroP": round(s["dev_calibrated"]["macro_p"], 4),
                 "dev_macroR": round(s["dev_calibrated"]["macro_r"], 4), "dev_acc": round(s["dev_calibrated"]["accuracy"], 4),
                 "test_pred1": s.get("test_share_pred_1")})
RES = pd.DataFrame(rows).sort_values("dev_macroF1", ascending=False)
RES.to_csv(f"{WORK}/RESULTS.csv", index=False)
display(RES)
for p in glob.glob(f"{OUT_DIR}/ENS*/ensemble_summary.json"):
    e = json.load(open(p)); print(f"{e['run']}: members {e['members']} | dev macro-F1 {e['dev_calibrated']['macro_f1']:.4f} at threshold {e['threshold']}")

## Per-class results of every run (dev, calibrated threshold)

In [ ]:
for r in RES.run:
    rep = json.load(open(f"{OUT_DIR}/{r}/dev_report.json"))
    print(format_report(rep["calibrated"], f"--- {r} (threshold {rep['threshold']})"), "\n")

## Section analysis (saved by the main notebook)

In [ ]:
for p in sorted(glob.glob(f"{OUT_DIR}/*/section_analysis.json")):
    s = json.load(open(p)); print(s["run"]); display(pd.DataFrame(s["views"]))
    att = s.get("chunk_attention_by_position(uniform 8 chunks)")
    if att:
        plt.figure(figsize=(7, 2.8)); plt.bar(list(att), list(att.values())); plt.axhline(1, color="k", ls=":")
        plt.title(f"{s['run']}: chunk attention by relative position (1 = uniform)"); plt.xticks(rotation=30, fontsize=8)
        plt.tight_layout(); plt.savefig(f"{WORK}/section_attention_{s['run']}.png", dpi=120); plt.show()

## Error analysis of the best single run (or set `RUN`)

In [ ]:
RUN = RES.iloc[0]["run"]
rep = json.load(open(f"{OUT_DIR}/{RUN}/dev_report.json"))
print(format_report(rep["calibrated"], f"{RUN}: dev at threshold {rep['threshold']}"))
P = pd.DataFrame(json.load(open(f"{OUT_DIR}/{RUN}/dev_predictions.json")))
P["kind"] = np.select([(P.gold == 1) & (P.pred == 1), (P.gold == 0) & (P.pred == 0), (P.gold == 0) & (P.pred == 1)], ["TP", "TN", "FP"], "FN")
P["correct"] = P.gold == P.pred
print("\nby outcome type (token length, confidence):")
display(P.groupby("kind").agg(cases=("id", "size"), median_tokens=("n_tokens", "median"), mean_tokens=("n_tokens", "mean"),
                              mean_prob1=("prob1", "mean")))
P["len_bin"] = pd.qcut(P.n_tokens, 5, duplicates="drop")
print("accuracy by judgment length (quintiles):")
display(P.groupby("len_bin", observed=True).agg(cases=("id", "size"), accuracy=("correct", "mean"), share_gold_1=("gold", "mean"),
                                                share_pred_1=("pred", "mean")))
P["conf_bin"] = pd.cut((P.prob1 - 0.5).abs(), [0, 0.1, 0.2, 0.3, 0.5], include_lowest=True)
print("accuracy by confidence |P(1) - 0.5|:")
display(P.groupby("conf_bin", observed=True).agg(cases=("id", "size"), accuracy=("correct", "mean")))
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for g in (0, 1):
    ax[0].hist(P.prob1[P.gold == g], bins=30, alpha=0.6, label=f"gold {g}")
ax[0].axvline(rep["threshold"], color="k", ls="--"); ax[0].legend(); ax[0].set_title(f"{RUN}: P(label 1) on dev")
for r in RES.run:
    h = json.load(open(f"{OUT_DIR}/{r}/history.json"))
    ax[1].plot([x["epoch"] for x in h], [x["train_loss"] for x in h], "o-", label=r, ms=3)
    ax[2].plot([x["epoch"] for x in h], [x["dev_macro_f1"] for x in h], "o-", label=r, ms=3)
ax[1].set_title("train loss"); ax[2].set_title("dev macro-F1 (argmax)"); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{WORK}/analysis_{RUN}.png", dpi=120); plt.show()

In [ ]:
dd = [p.parent.parent for p in Path(ROOT).rglob("dev.json") if p.parent.name == "Train"]
DATA_DIR = str(dd[0]) if dd else os.environ.get("P4_DATA_DIR", "")
if not DATA_DIR:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        DATA_DIR = str(D.download(f"{WORK}/p4data"))
    except Exception as e:
        print("dev.json not available, examples skipped:", e)
if DATA_DIR:
    dev_text = {c["id"]: c["text"] for c in D.load_split(DATA_DIR, "dev")}
    for kind in ("FP", "FN"):
        sub = P[P.kind == kind].assign(conf=lambda d: (d.prob1 - 0.5).abs()).sort_values("conf", ascending=False).head(3)
        for _, r in sub.iterrows():
            t = dev_text.get(r.id, "")
            print(f"\n--- {kind} {r.id} | gold {r.gold} | P(1) {r.prob1:.3f} | {r.n_tokens} tokens\n  START: {t[:300]}\n  END:   ...{t[-400:]}")

## Efficiency and knowledge distillation

In [ ]:
E = RES[["run", "arch", "method", "layers", "params_M", "trainable_M", "trainable_%", "peak_mem_GB", "train_min", "docs_per_s", "dev_macroF1"]].copy()
E["F1_vs_best"] = (E.dev_macroF1 - E.dev_macroF1.max()).round(4)
display(E.sort_values("params_M"))
E.to_csv(f"{WORK}/EFFICIENCY.csv", index=False)
for r, s in SUMS.items():
    t = s.get("kd_teacher")
    if t:
        print(f"KD {r}: student {s['params_total']/1e6:.1f}M params, {s['infer_docs_per_sec']} docs/s, dev F1 {s['dev_calibrated']['macro_f1']:.4f} | "
              f"teacher {t['teacher_backbone']} {(t['teacher_params'] or 0)/1e6:.1f}M params, dev F1 (argmax) {t['teacher_dev_macro_f1']:.4f}")
print("\nfiles written:", sorted(f for f in os.listdir(WORK) if f.endswith((".csv", ".png"))))